In [4]:
import sys
import numpy as np
import pandas as pd
from datetime import datetime
from sgp4.api import Satrec
from astropy.time import Time
from astropy.coordinates import TEME, ITRS, CartesianRepresentation, CartesianDifferential
from astropy.utils.iers import conf
import astropy.units as u

conf.auto_download = False 
sys.path.append('../src')
from sensor import GSENSE4040
from geometry import calculate_tangent_point_ecef
from atmosphere import get_atmospheric_params

H_SAT_APPROX = 500.0
MIN_TARGET_H = 20.0
FOCAL_LENGTH = 100.0

sensor = GSENSE4040(focal_length_mm=FOCAL_LENGTH)

s = '1 42017U 17008C   26114.50000000  .00000000  00000-0  50000-4 0  9998'
t = '2 42017  97.5000 100.0000 0010000   0.0000 180.0000 15.20000000500003'
satellite = Satrec.twoline2rv(s, t)

start_time = Time('2026-04-26T12:00:00', scale='utc')
time_steps = start_time + np.arange(0, 40) * 30 * u.second

results = []
print("正在执行全链路仿真 (轨道动力学 + 闭环姿态跟瞄 + NRLMSISE-00 环境解析)...")

for t_current in time_steps:
    jd1, jd2 = t_current.jd1, t_current.jd2
    e, r_teme, v_teme = satellite.sgp4(jd1, jd2)
    if e != 0: continue 
        
    teme_rep = CartesianRepresentation(
        r_teme * u.km, differentials=CartesianDifferential(v_teme * u.km / u.s)
    )
    teme_coord = TEME(teme_rep, obstime=t_current)
    itrs_coord = teme_coord.transform_to(ITRS(obstime=t_current))
    
    pos_ecef = [itrs_coord.x.value, itrs_coord.y.value, itrs_coord.z.value]
    vel_ecef = [
        itrs_coord.velocity.d_x.value, 
        itrs_coord.velocity.d_y.value, 
        itrs_coord.velocity.d_z.value
    ]
    
    sat_lat = itrs_coord.earth_location.lat.value
    sat_lon = itrs_coord.earth_location.lon.value
    current_sat_alt = itrs_coord.earth_location.height.to_value(u.km)
    
    # --- 1. 姿态闭环 ---
    dynamic_alpha = sensor.calc_center_angle_from_bottom_height(current_sat_alt, MIN_TARGET_H)
    tp_data = calculate_tangent_point_ecef(pos_ecef, vel_ecef, dynamic_alpha)
    
    # --- 2. 物理环境解算 ---
    # 将 Astropy Time 转换为标准的 Python datetime 对象
    dt_utc = t_current.datetime
    
    # 算卫星本体环境
    sat_env = get_atmospheric_params(dt_utc, current_sat_alt, sat_lat, sat_lon)
    # 算目标切点环境
    target_env = get_atmospheric_params(dt_utc, tp_data['h_tg'], tp_data['lat'], tp_data['lon'])
    
    # --- 3. 数据装配 ---
    results.append({
        'UTC_Time': t_current.isot,
        'Sat_Lon': round(sat_lon, 2),
        'Sat_Lat': round(sat_lat, 2),
        'Sat_Alt_km': round(current_sat_alt, 2),
        'Pitch_Deg': round(dynamic_alpha, 3),
        
        # 客户关心的卫星本体环境
        'Sat_Temp_K': round(sat_env['Temp_K'], 2),
        'Sat_Press_Pa': f"{sat_env['Pressure_Pa']:.2e}",
        
        # 光学探测目标点数据
        'Target_Lon': round(tp_data['lon'], 2),
        'Target_Lat': round(tp_data['lat'], 2),
        'Target_Alt_km': round(tp_data['h_tg'], 2),
        'Target_Temp_K': round(target_env['Temp_K'], 2),
        'Target_Press_Pa': f"{target_env['Pressure_Pa']:.2e}"
    })

df_results = pd.DataFrame(results)

# 强制显示所有列，方便审查
pd.set_option('display.max_columns', None)

# 落盘保存为交付物
csv_filename = 'integrated_mission_data.csv'
df_results.to_csv(csv_filename, index=False)

print(f"\n✅ 仿真完成！数据已落盘至 {csv_filename}。前 5 行预览：")
display(df_results.head(5))

正在执行全链路仿真 (轨道动力学 + 闭环姿态跟瞄 + NRLMSISE-00 环境解析)...

✅ 仿真完成！数据已落盘至 integrated_mission_data.csv。前 5 行预览：


,UTC_Time,Sat_Lon,Sat_Lat,Sat_Alt_km,Pitch_Deg,Sat_Temp_K,Sat_Press_Pa,Target_Lon,Target_Lat,Target_Alt_km,Target_Temp_K,Target_Press_Pa
0,2026-04-26T12:00:00.000,74.46,-42.84,516.80,11.334,1111.07,5.42e-07,58.97,-44.18,379.83,1109.87,4.23e-06
1,2026-04-26T12:00:30.000,73.89,-40.97,515.89,11.315,1116.77,5.61e-07,58.89,-42.33,380.06,1115.62,4.28e-06
2,2026-04-26T12:01:00.000,73.34,-39.10,515.00,11.296,1122.50,5.81e-07,58.78,-40.48,380.29,1121.35,4.33e-06
3,2026-04-26T12:01:30.000,72.82,-37.22,514.11,11.278,1128.22,6.01e-07,58.66,-38.63,380.52,1127.05,4.38e-06
4,2026-04-26T12:02:00.000,72.31,-35.34,513.23,11.260,1133.91,6.22e-07,58.53,-36.78,380.74,1132.69,4.43e-06
